In [15]:
import os 
import glob
import pandas as pd

from vip_slap2_analysis.utils.reorganize_slap2_session import (
    build_reorganization_plan,
    validate_plan,
    summarize_plan,
    execute_plan,
    write_report,
)

from pathlib import Path

In [2]:
basepath = r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics"
summary_path = os.path.join(basepath,'VIP_SD_summary.xlsx')
session_df = pd.read_excel(summary_path,sheet_name='sessions')

In [3]:
target_mouse = 826033

In [4]:
session_paths = session_df[session_df['subject_id']==target_mouse]['session_dir'].values[3:]

In [5]:
session_paths = [Path(p) for p in session_paths]
session_paths

[WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/2026-02-19_826033/826033_2026-02-19_13-47-57'),
 WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/2026-02-21_826033/826033_2026-02-21_09-23-34'),
 WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/2026-02-23_826033/826033_2026-02-23_10-45-21'),
 WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/2026-02-24_826033/826033_2026-02-24_14-14-45'),
 WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/2026-02-25_826033/826033_2026-02-25_08-49-29'),
 WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/2026-02-26_826033/826033_2026-02-26_12-40-54'),
 WindowsPath('//allen/aind/scratch/ophys/Andrew/VIP_synaptic_dynamics/iGluSnFR4f+RCaMP3/826033/2026-02-27_826033/826033_2026-02-27_13-53-35')]

In [6]:
example_manifest_tsv = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\example_tree_manifest.tsv")

In [17]:
all_plans = []
all_errors = {}

for sess_path in session_paths:
    print("=" * 100)
    print(sess_path)

    paths = []

    root = sess_path

    for p in root.rglob("*"):
        rel = p.relative_to(root)
        paths.append((str(rel), "dir" if p.is_dir() else "file"))
        
    out = root / "target_tree_manifest.tsv"
    out.write_text("\n".join(f"{t}\t{kind}" for t, kind in sorted(paths)))
    print("Wrote:", out)
    
    target_manifest_tsv = Path(root / "target_tree_manifest.tsv")
    
    try:
        plan = build_reorganization_plan(
            target_session_dir=sess_path,
            example_manifest_tsv=example_manifest_tsv,
            target_manifest_tsv=target_manifest_tsv,   # optional
            mouse_id=None,   # usually parse from folder name
        )

        print(summarize_plan(plan))

        errors = validate_plan(plan)
        if errors:
            all_errors[str(sess_path)] = errors
            print("Validation errors:")
            for err in errors:
                print("  -", err)
            continue

        all_plans.append(plan)

    except Exception as e:
        all_errors[str(sess_path)] = [str(e)]
        print(f"FAILED: {e}")

\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-19_826033\826033_2026-02-19_13-47-57
Wrote: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-19_826033\826033_2026-02-19_13-47-57\target_tree_manifest.tsv
Target session: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-19_826033\826033_2026-02-19_13-47-57
Raw root:       \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-19_826033\826033_2026-02-19_13-47-57
Processed root: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-19_826033\826033_2026-02-19_13-47-57_slap2_2026-02-19_13-47-57
Backup root:    \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-19_826033\slap2_826033_2026-02-19_13-47-57_remaining_data_backup
Planned moves:  1
  raw:          0
  processed:    0
  backup:       1
Warnings:
  - Processe

Wrote: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-27_826033\826033_2026-02-27_13-53-35\target_tree_manifest.tsv
Target session: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-27_826033\826033_2026-02-27_13-53-35
Raw root:       \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-27_826033\826033_2026-02-27_13-53-35
Processed root: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-27_826033\826033_2026-02-27_13-53-35_slap2_2026-02-27_13-53-35
Backup root:    \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-27_826033\slap2_826033_2026-02-27_13-53-35_remaining_data_backup
Planned moves:  129
  raw:          0
  processed:    127
  backup:       2


In [18]:
from pathlib import Path

executed_plans = []
execution_errors = {}

for plan in all_plans:
    print("=" * 100)
    print(f"Executing: {plan.target_session_dir}")

    errors = validate_plan(plan)
    if errors:
        execution_errors[str(plan.target_session_dir)] = errors
        print("Skipped due to validation errors:")
        for err in errors:
            print("  -", err)
        continue

    try:
        execute_plan(plan, execute=True)

        report_path = plan.target_session_dir.parent / f"{plan.target_session_dir.name}_reorganization_executed_report.tsv"
        write_report(plan, report_path)

        executed_plans.append(plan)
        print(f"Done. Report: {report_path}")

    except Exception as e:
        execution_errors[str(plan.target_session_dir)] = [str(e)]
        print(f"FAILED: {e}")

Executing: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-19_826033\826033_2026-02-19_13-47-57
Done. Report: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-19_826033\826033_2026-02-19_13-47-57_reorganization_executed_report.tsv
Executing: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-21_826033\826033_2026-02-21_09-23-34
Done. Report: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-21_826033\826033_2026-02-21_09-23-34_reorganization_executed_report.tsv
Executing: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-23_826033\826033_2026-02-23_10-45-21
Done. Report: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\2026-02-23_826033\826033_2026-02-23_10-45-21_reorganization_executed_report.tsv
Executing: \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynami